##This notebook calibrates and validates cGRHyMoLAP using the CAMELS-GB dataset. The models simulate streamflow ($Q$) from precipitation ($P$) and potential evapotranspiration (PET).

# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize # USE IN THE MODEL CALIBRATION

from google.colab import files
import zipfile
import os

from numba import njit
import warnings
warnings.filterwarnings('ignore')

import time

##CAMELS-DATA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

data_dir = "/content/drive/MyDrive/CAMELS_GB/"
start_date, end_date = '1990-01-01', '2014-12-31'

print("Loading, filtering, and aligning CSV files...\n")

# 1. Fast Load & Filter
df_pcp = pd.read_csv(f"{data_dir}pcp_mm.csv", index_col=0).loc[start_date:end_date]
df_pet = pd.read_csv(f"{data_dir}pet_mm.csv", index_col=0).loc[start_date:end_date]
df_q   = pd.read_csv(f"{data_dir}q_mmd_obs.csv", index_col=0).loc[start_date:end_date]

# 2. Fast Alignment
common_stations = sorted(list(set(df_pcp.columns) & set(df_pet.columns) & set(df_q.columns)))
df_pcp = df_pcp[common_stations]
df_pet = df_pet[common_stations]
df_q   = df_q[common_stations]

print(f"✅ Data aligned. Total common stations: {len(common_stations)}")

# ============================================
# Memory-Optimized Wrapper Classes
# ============================================
class SimpleArray:
    __slots__ = ['array']  # Prevents massive RAM bloat
    def __init__(self, array):
        self.array = array
    def to_numpy(self):
        return self.array

class StationData:
    __slots__ = ['data']
    def __init__(self, data_dict):
        self.data = data_dict
    def sel(self, dynamic_features=None):
        return SimpleArray(self.data[dynamic_features])

# ============================================
# Ultra-Fast Dictionary Construction
# ============================================
print("Building ds_recent dictionary...\n")

# Convert DataFrames to 2D NumPy matrices once (orders of magnitude faster than column iteration)
pcp_arr = df_pcp.to_numpy()
pet_arr = df_pet.to_numpy()
q_arr   = df_q.to_numpy()
date_arr = pd.to_datetime(df_pcp.index).to_numpy()

# Build the dictionary using fast index slicing
ds_recent = {
    st: StationData({
        'pcp_mm': pcp_arr[:, i],
        'pet_mm': pet_arr[:, i],
        'q_mmd_obs': q_arr[:, i],
        'date': date_arr
    })
    for i, st in enumerate(common_stations)
}

print(f"✅ Dictionary built successfully!")

# ============================================
# Validation Test
# ============================================
test_station = common_stations[0]
print(f"\nTesting data access for station: {test_station}")

Q_obs = ds_recent[test_station].sel(dynamic_features="q_mmd_obs").to_numpy()
P     = ds_recent[test_station].sel(dynamic_features="pcp_mm").to_numpy()
PET   = ds_recent[test_station].sel(dynamic_features="pet_mm").to_numpy()

print(f"✅ Extraction works correctly!")
print(f"   Q_obs shape: {Q_obs.shape}")
print(f"   Q_obs - min: {np.nanmin(Q_obs):.2f}, max: {np.nanmax(Q_obs):.2f}, mean: {np.nanmean(Q_obs):.2f}")
print(f"   Q_obs Missing: {np.sum(np.isnan(Q_obs))} ({np.sum(np.isnan(Q_obs))/len(Q_obs)*100:.1f}%)")

Loading, filtering, and aligning CSV files...

✅ Data aligned. Total common stations: 671
Building ds_recent dictionary...

✅ Dictionary built successfully!

Testing data access for station: 10002
✅ Extraction works correctly!
   Q_obs shape: (9131,)
   Q_obs - min: 0.21, max: 25.04, mean: 1.37
   Q_obs Missing: 0 (0.0%)


##Metrics

In [ ]:
@njit
def NSE(obs, sim):

    n = len(obs)

    mean_obs = 0.0
    count = 0

    for i in range(n):

        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            mean_obs += obs[i]
            count += 1

    if count == 0:
        return np.nan

    mean_obs /= count

    num = 0.0
    den = 0.0

    for i in range(n):

        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            num += (sim[i] - obs[i])**2
            den += (obs[i] - mean_obs)**2

    if den == 0:
        return np.nan

    return 1.0 - num / den

import numpy as np
from numba import njit


@njit
def RMSE(obs, sim):

    n = len(obs)

    mse = 0.0
    count = 0

    for i in range(n):

        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            diff = sim[i] - obs[i]
            mse += diff * diff
            count += 1

    if count == 0:
        return np.nan

    return np.sqrt(mse / count)


@njit
def FHV(obs, sim, top_fraction=0.02):

    mask = ~np.isnan(obs) & ~np.isnan(sim)

    obs = obs[mask]
    sim = sim[mask]

    if len(obs) == 0:
        return np.nan

    n_top = int(len(obs) * top_fraction)

    if n_top == 0:
        return np.nan

    idx = np.argsort(obs)[-n_top:]

    return np.sum(sim[idx] - obs[idx]) / np.sum(obs[idx])


@njit
def FLV(obs, sim, bottom_fraction=0.3):
    epsilon = 1e-6
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_bot = int(len(obs) * bottom_fraction)
    if n_bot == 0:
        return np.nan
    idx = np.argsort(obs)[:n_bot]
    obs_bot = obs[idx]
    sim_bot = sim[idx]
    return  np.sum(sim_bot - obs_bot) / (np.sum(obs_bot) + epsilon)


def KGE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]

    r = np.corrcoef(obs, sim)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)

    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2), r, alpha, beta

In [ ]:
## General Parameters of the code
b1_ratio = 0.7
max_missing_ratio = 0.1

stations = list(ds_recent.keys())

## cGRHyMoLAP

#Q(t+1) from t+1 imputs

In [ ]:
start_time = time.perf_counter()

# ============================================
# TIME DISCRETIZATION
# ============================================
dt = 0.5
nsub = int(1.0 / dt)

# ============================================
# COUPLED STORAGE + RUNOFF (S AND Q TOGETHER)
# ============================================


@njit
def cGRHyMoLAP_Model(params, Q0, Pn, En):

    MU, LAMBDA, X1, GAMMA = params

    n = len(Pn)

    Q = np.zeros(n)
    S = np.zeros(n)

    Q[0] = Q0
    S[0] = X1 / 2.0

    ratio = 4.0 / 9.0
    c = (1.0 / (4.0 * X1**4)) * ratio**4

    for t in range(n-1):

        Q_day = Q[t]
        S_day = S[t]

        for k in range(nsub):
            # choose forcing
            if k == nsub - 1:
                P_n = Pn[t + 1]
                E_n = En[t + 1]
            else:
                P_n = Pn[t]
                E_n = En[t]

            temp = S_day / X1

            Ps = P_n * (1.0 - temp**2)
            Es = E_n * (2.0 * temp - temp**2)

            perc = c * S_day**5

            # update S (first ODE)
            S_day = S_day + dt * (Ps - Es - perc)
            if S_day < 0:
                S_day = 0.0

            dQ = (
                -(MU / LAMBDA) * Q_day**(2 * MU - 1)
                + GAMMA * (c * S_day**5) * P_n
            )

            Q_day = Q_day + dt * dQ
            if Q_day < 0:
                Q_day = 0.0

        S[t + 1] = S_day
        Q[t + 1] = Q_day

    return Q, S


# ============================================
# MAIN LOOP (UNCHANGED STRUCTURE)
# ============================================

results_cGRH_semi_exp = {}

for i, station_id in enumerate(stations):

    print(f"\n=== Station {station_id} ===, Number = {i+1}")

    Q_obs = ds_recent[station_id].sel(
        dynamic_features="q_mmd_obs"
    ).to_numpy()

    P = ds_recent[station_id].sel(
        dynamic_features="pcp_mm"
    ).to_numpy()

    PET = ds_recent[station_id].sel(
        dynamic_features="pet_mm"
    ).to_numpy()

    Pn = np.maximum(0, P - PET)
    En = np.maximum(0, PET - P)

    N = len(Q_obs)

    if N == 0 or np.all(np.isnan(Q_obs)):
        continue

    if np.sum(np.isnan(Q_obs)) / N > max_missing_ratio:
        continue

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    if np.isnan(Q0):
       Q0 = 0

    # ============================================
    # OPTIMIZATION
    # ============================================
    grhymolap_param_bound = [(0.5, 3.5), (1e-3, 300.0), (1e-3, 5000.0), (1e-3, 20.0)]

    initial_guesses = [
    [0.5, 1.0, 80, 0.05],
    [0.7, 2.0, 120, 0.1],
    [1.0, 5.0, 150, 0.2],
    [1.2, 8.0, 250, 0.3],
    [1.5, 10.0, 300, 0.5],
    [2.0, 15.0, 400, 0.8],
    [2.5, 20.0, 600, 1.0],
    [0.8, 3.0, 800, 0.2],
    [1.0, 6.0, 1000, 0.4],
    [3.0, 25.0, 200, 0.1],
    [1.0, 12.0, 1500, 0.6]
]

    best_res = None
    best_val = float("inf")

    for guess in initial_guesses:

        res = minimize(
            lambda p: RMSE(
                Q_obs[:b1],
                cGRHyMoLAP_Model(p, Q0, Pn[:b1], En[:b1])[0]
            ),
            guess,
            method="Nelder-Mead",
            bounds = grhymolap_param_bound,
            options={'maxiter': 2500, 'disp': False}
        )

        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    # ============================================
    # SIMULATION
    # ============================================

    Qsim, S = cGRHyMoLAP_Model(best_res.x, Q0, Pn, En)

    NSE_cal = NSE(Q_obs[:b1], Qsim[:b1])
    RMSE_cal = RMSE(Q_obs[:b1], Qsim[:b1])

    NSE_val = NSE(Q_obs[b1:], Qsim[b1:])
    RMSE_val = RMSE(Q_obs[b1:], Qsim[b1:])

    FHV_val = FHV(Q_obs[b1:], Qsim[b1:])
    FLV_val = FLV(Q_obs[b1:], Qsim[b1:])

    KGE_val, KGE_1, KGE_2, KGE_3  = KGE(Q_obs[b1:], Qsim[b1:])

    results_cGRH_semi_exp[station_id] = {

        "params": best_res.x,
        "Qsim": Qsim,
        "Q_obs": Q_obs,
        "S": S,

        "NSE_cal": NSE_cal,
        "NSE_val": NSE_val,
        "RMSE_cal": RMSE_cal,
        "RMSE_val": RMSE_val,

        "FHV_val": FHV_val,
        "FLV_val": FLV_val,
        "KGE_val": KGE_val,
        "KGE_1": KGE_1,
        "KGE_2": KGE_2,
        "KGE_3": KGE_3,

    }

    print(f"   NSE cal: {NSE_cal:.4f}")
    print(f"   NSE val: {NSE_val:.4f}")

print(f"\n✅ Simulation completed for {len(results_cGRH_semi_exp)} basins.")

end_time = time.perf_counter()

total_time = end_time - start_time

print(f"Total time: {total_time:.2f} s")
print(f"Total time: {total_time/60:.2f} min")
print(f"Total time: {total_time/3600:.2f} h")


=== Station 10002 ===, Number = 1
   NSE cal: 0.7389
   NSE val: 0.5976

=== Station 10003 ===, Number = 2
   NSE cal: 0.8359
   NSE val: 0.7533

=== Station 1001 ===, Number = 3

=== Station 101002 ===, Number = 4
   NSE cal: 0.7208
   NSE val: 0.7376

=== Station 101005 ===, Number = 5
   NSE cal: 0.7396
   NSE val: 0.7604

=== Station 102001 ===, Number = 6
   NSE cal: 0.7694
   NSE val: 0.7743

=== Station 106001 ===, Number = 7

=== Station 107001 ===, Number = 8

=== Station 11001 ===, Number = 9
   NSE cal: 0.7828
   NSE val: 0.6295

=== Station 11003 ===, Number = 10
   NSE cal: 0.6261
   NSE val: 0.4177

=== Station 11004 ===, Number = 11
   NSE cal: 0.7748
   NSE val: 0.6552

=== Station 12001 ===, Number = 12
   NSE cal: 0.5818
   NSE val: 0.4310

=== Station 12002 ===, Number = 13
   NSE cal: 0.6481
   NSE val: 0.5186

=== Station 12005 ===, Number = 14
   NSE cal: 0.6121
   NSE val: 0.5331

=== Station 12006 ===, Number = 15
   NSE cal: 0.4992
   NSE val: 0.3245

=== Stat

In [ ]:
# ============================================
# SAVE RESULTS TO CSV
# ============================================

rows = []

for station_id, res in results_cGRH_semi_exp.items():

    MU, LAMBDA, X1, GAMMA = res["params"]

    rows.append({
        "station_id": station_id,

        "MU": MU,
        "LAMBDA": LAMBDA,
        "X1": X1,
        "GAMMA": GAMMA,

        "NSE_cal": res["NSE_cal"],
        "NSE_val": res["NSE_val"],

        "RMSE_cal": res["RMSE_cal"],
        "RMSE_val": res["RMSE_val"],

        "FHV_val": res["FHV_val"],
        "FLV_val": res["FLV_val"],
        "KGE_val": res["KGE_val"],
    })

df_results = pd.DataFrame(rows)

csv_name = "cGRHyMoLAP_semi_exp_results.csv"
df_results.to_csv(csv_name, index=False)

print(f"✅ Results saved to {csv_name}")

# ============================================
# DOWNLOAD TO LOCAL COMPUTER
# ============================================

#files.download(csv_name)

✅ Results saved to cGRHyMoLAP_semi_exp_results.csv


In [ ]:
# =============================================================
# 📌 EXTRACTION OF NSE & RMSE — CALIBRATION
# =============================================================
nse_cal = [res['NSE_cal'] for res in results_cGRH_semi_exp.values() if not np.isnan(res['NSE_cal'])]
rmse_cal = [res['RMSE_cal'] for res in results_cGRH_semi_exp.values() if not np.isnan(res['RMSE_cal'])]

print("\n================= CALIBRATION =================\n")

# ----- NSE -----
if nse_cal:
    print(f"NSE Calibration -> Median: {np.percentile(nse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_cal, 95):.4f}")
    print("MEAN NSE_CAL:", np.mean(nse_cal))
    print("MIN NSE_CAL:", np.min(nse_cal))
    print("MAX NSE_CAL:", np.max(nse_cal))
else:
    print("No NSE available for the calibration.")

# ----- RMSE -----
print("\n--- RMSE Calibration ---")
if rmse_cal:
    print(f"RMSE Calibration -> Median: {np.percentile(rmse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(rmse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(rmse_cal, 95):.4f}")
    print("MEAN RMSE_CAL:", np.mean(rmse_cal))
    print("MIN RMSE_CAL:", np.min(rmse_cal))
    print("MAX RMSE_CAL:", np.max(rmse_cal))
else:
    print("No RMSE available for the calibration.")


# =============================================================
# 📌 EXTRACTION OF NSE & RMSE — VALIDATION
# =============================================================
print("\n\n================= VALIDATION =================\n")

nse_val = [res['NSE_val'] for res in results_cGRH_semi_exp.values() if not np.isnan(res['NSE_val'])]
rmse_val = [res['RMSE_val'] for res in results_cGRH_semi_exp.values() if not np.isnan(res['RMSE_val'])]

# ----- NSE -----
if nse_val:
    print(f"NSE Validation -> Median: {np.percentile(nse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_val, 95):.4f}")
    print("MEAN NSE_VAL:", np.mean(nse_val))
    print("MIN NSE_VAL:", np.min(nse_val))
    print("MAX NSE_VAL:", np.max(nse_val))
else:
    print("No valid station for NSE in validation.")

# ----- RMSE -----
print("\n--- RMSE Validation ---")
if rmse_val:
    print(f"RMSE Validation -> Median: {np.percentile(rmse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(rmse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(rmse_val, 95):.4f}")
    print("MEAN RMSE_VAL:", np.mean(rmse_val))
    print("MIN RMSE_VAL:", np.min(rmse_val))
    print("MAX RMSE_VAL:", np.max(rmse_val))
else:
    print("No valid station for RMSE in validation.")


================= CALIBRATION =================

NSE Calibration -> Median: 0.816, 5th percentile: 0.6339, 95th percentile: 0.9196
MEAN NSE_CAL: 0.8027562153567339
MIN NSE_CAL: 0.0729768934549112
MAX NSE_CAL: 0.9544865469433179

--- RMSE Calibration ---
RMSE Calibration -> Median: 0.781, 5th percentile: 0.1444, 95th percentile: 2.7130
MEAN RMSE_CAL: 1.0622916588005762
MIN RMSE_CAL: 0.047190860584618306
MAX RMSE_CAL: 10.223686620144784


================= VALIDATION =================

NSE Validation -> Median: 0.786, 5th percentile: 0.5338, 95th percentile: 0.9100
MEAN NSE_VAL: 0.7600383432354919
MIN NSE_VAL: -1.646626692073669
MAX NSE_VAL: 0.9561095821977753

--- RMSE Validation ---
RMSE Validation -> Median: 0.941, 5th percentile: 0.1630, 95th percentile: 3.0530
MEAN RMSE_VAL: 1.1945957890033891
MIN RMSE_VAL: 0.03941320076171775
MAX RMSE_VAL: 9.997008130116887
